# 06 机器学习势（MLFF）：从性质预测到原子模拟

> 🟣 **Level C · 了解即可** | 拓展阅读 | 完成标准：知道 MLFF 与普通 property model 的目标不同，并知道“预训练模型能运行”不等于“对目标 COF 可靠”。

普通 supervised ML 学习 `descriptors → property`；MLFF 更接近学习 `atomic configuration → energy / forces`。


## 1. 先回顾三种常见原子模拟思路

### 经典力场（classical force field）
用预先规定的数学形式和参数计算原子相互作用。优点是快，适合大体系和长时间 MD；缺点是适用范围受力场形式和参数限制。

### DFT / AIMD
从电子结构层面计算能量和力，通常更昂贵。AIMD 因为每一步都要做电子结构计算，所以时间和体系尺寸受到明显限制。

### MLFF
用大量高精度计算数据训练机器学习模型，让模型近似势能面，之后以更低成本预测能量和原子力。


## 2. 什么叫势能面？
给定一组原子位置，体系会对应一个能量。原子位置变化，能量也变化。

可以简单写成：`atomic positions → energy`。

原子力与能量随坐标的变化有关。因此 MLFF 训练数据常包含：
- energy（能量）；
- force（原子力）；
- 有时还包含 stress / virial（应力相关量）。

入门阶段只需要知道：**MLFF 不只是做普通性质回归，而是在学习原子运动所依赖的能量/力关系。**


## 3. 为什么研究者想用 MLFF？
理想目标是同时接近：
- DFT 级别的精度；
- 经典 MD 的速度和可扩展性。

实际能否做到，取决于训练数据是否覆盖目标体系会访问的结构空间。


## 4. 什么是 pretrained MLFF？
有些模型已经在大型材料数据库上训练完成，可以直接加载。这类似使用一个已经学过大量结构的基础模型。

但“可以运行”不等于“对你的 COF 可靠”。需要检查：
- 训练集中是否包含类似元素和化学环境；
- 是否覆盖层间作用、柔性结构、guest、水、离子等；
- 在你自己的 DFT reference structures 上 energy / force error 是否可接受。


In [ ]:
# 可选演示：仅展示如何调用预训练 CHGNet，不作为 COF 可靠性验证
!pip -q install chgnet pymatgen
from pymatgen.core import Lattice, Structure
from chgnet.model import CHGNet
structure=Structure(Lattice.cubic(4.2),['Li','Cl'],[[0,0,0],[0.5,0.5,0.5]])
model=CHGNet.load()
pred=model.predict_structure(structure)
print('Energy/atom:',pred['e'])
print('Forces shape:',pred['f'].shape)


## 5. 对 COF 特别要谨慎
COF 可能涉及：
- 层间 vdW / stacking / slip；
- framework flexibility；
- 水和 guest molecules；
- 离子；
- 极端构型；
- 某些情况下的成键变化或质子转移。

如果训练数据没有覆盖这些情况，预训练势可能在模拟中外推到不可靠区域。


## 6. 只需了解的研究级 workflow
`DFT reference data → train/fine-tune MLFF → independent test → exploratory MD → 检查异常/外推 → 增加 reference data → 再训练 → production MD`

这里列出流程只是为了让你知道后续研究会做什么，**本课程不要求你执行这整套流程。**


## 本章术语表
- force field：力场；
- DFT：密度泛函理论；
- AIMD：第一性原理分子动力学；
- potential energy surface：势能面；
- MLFF：机器学习势；
- pretrained model：预训练模型；
- fine-tuning：微调；
- active learning：主动学习。


## 本章只需回答三个问题
1. 为什么 DFT/AIMD 通常比经典 MD 贵？
2. MLFF 想解决什么计算成本问题？
3. 为什么预训练 MLFF 在新 COF 上仍然需要验证？

能回答这三个问题，就已经完成本章。


## 数据来源与扩展阅读
[Dataset contracts / 数据使用约定](../docs/data_resources.md) · [COFSpace](https://github.com/gokhanonderaksu/COFSpace) · [CURATED-COFs](https://github.com/danieleongari/CURATED-COFs)
